# YM HOTRG SU(3) Curvature Toy Pipeline

This Colab-ready notebook wires together:

- A toy SU(3) plaquette tensor (`ym_su3_tensor.py`)
- A basic HOTRG flow (`ym_hotrg.py`)
- A Jacobian scaffold using autodiff (`ym_hotrg_jacobian.py`)
- Curvature and Riccati utilities (`ym_curvature_flow.py`)

**Important:** This is not yet a full SU(3) irrep-expansion or manual HOTRG Jacobian.
It is a *scaffold* matching your architecture, with a working numerical toy model
you can scale up and refine on an A100.


In [ ]:
import jax, jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())


In [ ]:
# If running in Colab, upload the .py files from your local machine
# (ym_su3_tensor.py, ym_hotrg.py, ym_hotrg_jacobian.py, ym_curvature_flow.py)
# using the Colab file upload UI, or mount Google Drive and copy them.
import os
print('Current directory:', os.getcwd())


In [ ]:
import importlib
import ym_su3_tensor as su3
import ym_hotrg as hotrg
import ym_hotrg_jacobian as hotrg_jac
import ym_curvature_flow as cf
print('Modules imported.')


In [ ]:
beta = 2.0
chi = 8   # toy bond dimension; scale up later
n_steps = 3

T = su3.make_su3_plaquette_tensor_toy(beta)
print('Initial tensor shape:', T.shape)

T_rg = hotrg.hotrg_sweep(T, chi=chi, n_steps=n_steps)
print('Coarse tensor shape after', n_steps, 'steps:', T_rg.shape)


In [ ]:
H = su3.local_wilson_hessian_toy(jnp.eye(3, dtype=jnp.complex128), beta)
print('Initial Hessian shape:', H.shape)

H_final, lmins, lmaxs = cf.riccati_flow(H, eta=0.2, n_steps=10)
print('Final eigenvalue min/max:', lmins[-1], lmaxs[-1])

from ym_curvature_flow import convexity_radius, gamma2_curvature
rc0 = convexity_radius(lmins[0], lmaxs[0])
rcf = convexity_radius(lmins[-1], lmaxs[-1])
k0 = gamma2_curvature(lmins[0], lmaxs[0])
kf = gamma2_curvature(lmins[-1], lmaxs[-1])
print('Initial rc, kappa:', rc0, k0)
print('Final   rc, kappa:', rcf, kf)
